# 5) Market coupling: zones instead of nodes

## Objectives

At the end of this notebook you should be able to:

- clear a **zonal** market — one price per bidding zone, the network inside a zone ignored — and check the **real** flows its dispatch causes
- price the **redispatch** a transmission system operator needs when the market's dispatch overloads a line
- limit trade between zones with a **net transfer capacity** (NTC), and explain why no NTC can fix congestion inside a zone
- build **flow-based** constraints from zonal PTDFs, generation shift keys (GSKs) and a base case, and see where their approximations break
- weigh the designs by market cost, prices and physical security

Lesson 4 priced every node. Europe does not: the day-ahead market couples **bidding zones** — mostly whole countries — and the network inside a zone is the transmission system operator's problem after the market has cleared. This lesson splits lesson 4's network into two illustrative zones and asks what that simplification costs.

In [1]:
import logging
import warnings

import linopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")
logging.getLogger("linopy").setLevel(logging.ERROR)

### The network, the nodal benchmark and two zones

Lesson 4's network with 400 MW on every line. The **North** zone holds the load centre (Zagreb, Osijek) with coal and imports; the **South** zone holds wind at Velebit, hydro at Split and gas at Rijeka. The zones are illustrative — Croatia is one bidding zone — but the split mirrors a familiar pattern: cheap generation far from the load, like wind in the north of Germany and load in its south.

In [2]:
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM, LIMIT = 0.32, 400.0
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")
ZONE = pd.Series({"Zagreb": "North", "Osijek": "North", "Rijeka": "South", "Velebit": "South", "Split": "South"})[BUSES]  # bus order!
GENS["zone"] = GENS.bus.map(ZONE)
ZONE_LOAD = LOAD.groupby(ZONE).sum()


def ptdf(reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * LINES.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(LINES), len(BUSES)))
    A[range(len(LINES)), LINES.bus0.map(idx)] = 1
    A[range(len(LINES)), LINES.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=LINES.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1
GI = pd.Index(GENS.index, name="gen")
COST = xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[GI])


def real_flows(dispatch):
    """The physical line flows (MW) that a dispatch causes - whatever market produced it."""
    return pd.Series(PTDF.to_numpy() @ (G @ dispatch[GENS.index].to_numpy() - LOAD.to_numpy()), index=LINES.index)


def nodal():
    """Lesson 4's benchmark: the cheapest dispatch that keeps every line within its limit."""
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    for l, line in enumerate(LINES.index):
        flow = (p * xr.DataArray((PTDF.to_numpy() @ G)[l], coords=[GI])).sum() - float(PTDF.to_numpy()[l] @ LOAD.to_numpy())
        m.add_constraints(flow <= LIMIT, name=f"upper {line}")
        m.add_constraints(flow >= -LIMIT, name=f"lower {line}")
    m.add_objective((p * COST).sum())
    m.solve(solver_name="highs", output_flag=False)
    return {"cost": float(m.objective.value), "dispatch": p.solution.to_pandas()}


NODAL = nodal()
print(f"nodal benchmark: {NODAL['cost']:,.1f} EUR/h, max real flow {real_flows(NODAL['dispatch']).abs().max():.1f} MW")
print("zone loads (MW):", ZONE_LOAD.to_dict(), "| zone capacity (MW):", GENS.groupby('zone').p_max_MW.sum().to_dict())

nodal benchmark: 81,956.3 EUR/h, max real flow 400.0 MW
zone loads (MW): {'North': 1150, 'South': 750} | zone capacity (MW): {'North': 1200, 'South': 1900}


### One zone: the market ignores the network

A single bidding zone clears like Module 03's market: the cheapest plants run, one price for everyone. Then the transmission system operator checks what the dispatch does to its lines. If a line is overloaded, it **redispatches**: it orders plants on one side of the congestion down and plants on the other side up, paying the up-regulated plants their costs and recovering the costs saved by the down-regulated ones. Here, cost-based redispatch to the cheapest secure dispatch lands exactly on the nodal solution.

In [3]:
def zonal_market(ntc=None, flow_based=None):
    """Two-zone market. ntc: limit on the South-to-North exchange (MW); flow_based: list of (line, F_ref, k, E_ref).
    With neither, the two zones trade freely - one zone in effect."""
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    E = m.add_variables(lower=-1e5, upper=1e5, name="exchange")                   # South -> North, MW
    south = sum(p.sel(gen=g) for g in GENS.index[GENS.zone == "South"])
    north = sum(p.sel(gen=g) for g in GENS.index[GENS.zone == "North"])
    m.add_constraints(south - E == ZONE_LOAD["South"], name="balance South")
    m.add_constraints(north + E == ZONE_LOAD["North"], name="balance North")
    if ntc is not None:
        m.add_constraints(E <= ntc, name="ntc")
        m.add_constraints(E >= -ntc, name="ntc reverse")
    for line, f_ref, k, e_ref in flow_based or []:
        m.add_constraints(k * E <= LIMIT - f_ref + k * e_ref, name=f"fb upper {line}")
        m.add_constraints(k * E >= -LIMIT - f_ref + k * e_ref, name=f"fb lower {line}")
    m.add_objective((p * COST).sum())
    m.solve(solver_name="highs", output_flag=False)
    dispatch = p.solution.to_pandas()
    flows = real_flows(dispatch)
    return {"cost": float(m.objective.value), "dispatch": dispatch, "exchange": float(E.solution),
            "price": {z: abs(float(m.constraints[f"balance {z}"].dual)) for z in ("North", "South")},
            "flows": flows, "overloaded": list(flows.index[flows.abs() > LIMIT + 1e-6])}


one_zone = zonal_market()
redispatch = NODAL["dispatch"] - one_zone["dispatch"]
redispatch_cost = float((redispatch * GENS.cost_EUR_per_MWh).sum())
print(f"one zone: market cost {one_zone['cost']:,.0f} EUR/h, price {one_zone['price']['North']:.0f} EUR/MWh everywhere")
print(f"real flows: {one_zone['flows'].round(0).to_dict()}")
print(f"overloaded: {one_zone['overloaded']} ({one_zone['flows'].abs().max():.1f} MW on a {LIMIT:.0f} MW line)")
print(f"redispatch (MW): {redispatch[redispatch.abs() > 1e-6].round(1).to_dict()}; cost {redispatch_cost:,.1f} EUR/h")

one zone: market cost 71,000 EUR/h, price 90 EUR/MWh everywhere
real flows: {'Zagreb-Rijeka': -391.0, 'Rijeka-Velebit': -491.0, 'Velebit-Split': 109.0, 'Zagreb-Split': -273.0, 'Zagreb-Osijek': -236.0, 'Osijek-Split': -86.0}
overloaded: ['Rijeka-Velebit'] (490.9 MW on a 400 MW line)
redispatch (MW): {'wind Velebit': -115.0, 'gas Rijeka': 100.0, 'import Zagreb': 15.0}; cost 10,956.3 EUR/h


As one zone, the market clears at **71,000 €/h with one price, 90 €/MWh**, everywhere — Module 03's merit order, blind to the network. Its dispatch puts **491 MW on Rijeka–Velebit**, a 400 MW line: Velebit's full 600 MW of wind has nowhere to go but through Rijeka. The transmission system operator must redispatch: 115 MW of wind down, 100 MW of gas at Rijeka and 15 MW of imports at Zagreb up. That costs **10,956 €/h — exactly the gap to the nodal dispatch**, because cost-based redispatch to the cheapest secure dispatch arrives at the nodal solution. The zonal market is cheaper only on paper: society pays the same 81,956 €/h for a secure system either way, but the redispatch part is paid by the operator and passed on through network tariffs, while the market price — 90 €/MWh at Velebit and at Zagreb alike — tells nobody where power is scarce and where it is stranded.

### Two zones and a net transfer capacity

Now split the market at the North–South border and let the two zones trade up to an **NTC** — a single number, set by the transmission system operators in advance, for how much power may cross the border. Sweep it:

In [4]:
rows = {}
for ntc in [1000, 600, 500, 400, 300]:
    r = zonal_market(ntc=ntc)
    rows[ntc] = {"market_cost_EUR_per_h": r["cost"], "price_North": r["price"]["North"], "price_South": r["price"]["South"],
                 "exchange_MW": r["exchange"], "Rijeka-Velebit_real_MW": abs(r["flows"]["Rijeka-Velebit"]),
                 "overloaded": ", ".join(r["overloaded"]) or "-"}
ntc_sweep = pd.DataFrame(rows).T.rename_axis("NTC (MW)")
ntc_sweep

         market_cost_EUR_per_h price_North price_South exchange_MW  \
NTC (MW)                                                             
1000                   71000.0        90.0        90.0       750.0   
600                    77000.0       130.0        90.0       600.0   
500                    84000.0       130.0        30.0       500.0   
400                    94000.0       130.0        30.0       400.0   
300                   104000.0       130.0        30.0       300.0   

         Rijeka-Velebit_real_MW      overloaded  
NTC (MW)                                         
1000                 490.945748  Rijeka-Velebit  
600                   523.11217  Rijeka-Velebit  
500                  516.825513  Rijeka-Velebit  
400                  482.807918  Rijeka-Velebit  
300                  448.790323  Rijeka-Velebit  

Below the 750 MW the zones would trade freely, the NTC binds: market cost rises from 71,000 to 104,000 €/h and the zonal prices split, 130 €/MWh in the North against 90 and then 30 €/MWh in the South. And the overload on Rijeka–Velebit **never goes away** — at an NTC of 600 MW it even gets worse, 523 MW instead of 491. The reason is visible in the dispatch: to export less, the South must produce less, and the zonal market cuts the South's most expensive plant — gas at Rijeka, from 200 to 50 MW, then to zero, and finally hydro at Split. Wind at Velebit stays at its full 600 MW, because it is the South's cheapest plant. But it is the one plant whose output loads Rijeka–Velebit; cutting gas at Rijeka only means that more of Rijeka's own load is served by wind across the congested line. The NTC sits on the wrong border: the congestion is **inside** the South zone, and a zonal market cannot see where inside a zone power is produced. At an NTC of 400 MW the market pays 94,000 €/h — 12,000 €/h more than the nodal optimum — and still needs redispatch.

### Flow-based market coupling

Since 2015 the Central Western European markets — and since 2022 the Core region, which includes Croatia — replace NTCs with **flow-based** constraints. The market still sees only zones, but each monitored line (a critical network element) gets a constraint on the zonal exchange:

flow_ℓ = F_ref,ℓ + k_ℓ · (E − E_ref) ≤ limit

- the **base case**: a forecast of the network two days ahead, with flows F_ref at an exchange E_ref — here, the one-zone dispatch;
- the **zonal PTDF** k_ℓ: how much line ℓ's flow changes per MW exported from South to North. It needs an assumption about **where in each zone** the extra MW is produced: the **generation shift keys** (GSK). Here, shares of each zone's generation capacity.

In [5]:
capacity = GENS.groupby("bus").p_max_MW.sum().reindex(BUSES).fillna(0)
GSK = capacity / capacity.groupby(ZONE).transform("sum")
zonal_ptdf = {z: PTDF.to_numpy() @ np.where(ZONE == z, GSK, 0.0) for z in ("North", "South")}
K = pd.Series(zonal_ptdf["South"] - zonal_ptdf["North"], index=LINES.index)       # flow per MW South -> North
E_REF, F_REF = one_zone["exchange"], one_zone["flows"]
print("GSK:", GSK.round(3).to_dict())
print("zonal PTDF of the exchange (MW per MW):", K.round(3).to_dict())

cross_border = [l for l in LINES.index if ZONE[LINES.bus0[l]] != ZONE[LINES.bus1[l]]]
designs = {"flow-based, cross-border lines": cross_border, "flow-based, all lines": list(LINES.index)}
rows = {}
for label, monitored in designs.items():
    r = zonal_market(flow_based=[(l, F_REF[l], K[l], E_REF) for l in monitored])
    predicted = F_REF["Rijeka-Velebit"] + K["Rijeka-Velebit"] * (r["exchange"] - E_REF)
    rows[label] = {"market_cost_EUR_per_h": r["cost"], "price_North": r["price"]["North"], "price_South": r["price"]["South"],
                   "exchange_MW": r["exchange"], "Rijeka-Velebit_predicted_MW": abs(predicted),
                   "Rijeka-Velebit_real_MW": abs(r["flows"]["Rijeka-Velebit"]), "overloaded": ", ".join(r["overloaded"]) or "-"}
    if label.endswith("all lines"):
        fb_all = r
flow_based = pd.DataFrame(rows).T
print("cross-border lines:", cross_border)
flow_based

GSK: {'Zagreb': 0.667, 'Rijeka': 0.316, 'Split': 0.368, 'Osijek': 0.333, 'Velebit': 0.316}
zonal PTDF of the exchange (MW per MW): {'Zagreb-Rijeka': -0.515, 'Rijeka-Velebit': -0.2, 'Velebit-Split': 0.116, 'Zagreb-Split': -0.22, 'Zagreb-Osijek': 0.068, 'Osijek-Split': -0.265}


cross-border lines: ['Zagreb-Rijeka', 'Zagreb-Split', 'Osijek-Split']


                               market_cost_EUR_per_h price_North price_South  \
flow-based, cross-border lines               71000.0        90.0        90.0   
flow-based, all lines                  104586.035717       130.0        30.0   

                               exchange_MW Rijeka-Velebit_predicted_MW  \
flow-based, cross-border lines       750.0                  490.945748   
flow-based, all lines           294.139643                       400.0   

                               Rijeka-Velebit_real_MW      overloaded  
flow-based, cross-border lines             490.945748  Rijeka-Velebit  
flow-based, all lines                       446.79677  Rijeka-Velebit  

Monitoring only the three cross-border lines changes nothing: none of them is congested, so the flow-based market reproduces the one-zone result. Monitoring all lines brings Rijeka–Velebit into the market — through the only lever the market has, the exchange. One MW less exported from South to North lowers the predicted flow on Rijeka–Velebit by only 0.2 MW, so removing 91 MW of overload takes **455 MW less exchange**: the market cuts it from 750 to 294 MW, pays **104,586 €/h** — 22,630 €/h above the nodal optimum — and opens a gap of 100 €/MWh between the zones.

And the line is **still overloaded**: the market predicted exactly 400 MW, the real flow is 447 MW. The prediction rests on the GSK, which assumes that the South's lower output is shared by its plants in proportion to their capacity — 31.6 % of it at Velebit. The market instead cut hydro and gas and kept wind at full output, as in the NTC case. The approximation is exactly wrong where it matters. This is the dilemma behind the European debate on internal lines in flow-based coupling: leaving them out keeps the market efficient but blind to them; putting them in makes the market expensive and still does not make the network secure.

### The designs side by side

In [6]:
summary = pd.DataFrame({
    "nodal": {"market_cost": NODAL["cost"], "Rijeka-Velebit_real": abs(real_flows(NODAL["dispatch"])["Rijeka-Velebit"])},
    "one zone": {"market_cost": one_zone["cost"], "Rijeka-Velebit_real": abs(one_zone["flows"]["Rijeka-Velebit"])},
    "NTC 400 MW": {"market_cost": ntc_sweep.loc[400, "market_cost_EUR_per_h"],
                   "Rijeka-Velebit_real": ntc_sweep.loc[400, "Rijeka-Velebit_real_MW"]},
    "flow-based, all lines": {"market_cost": fb_all["cost"], "Rijeka-Velebit_real": abs(fb_all["flows"]["Rijeka-Velebit"])}}).T
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
colours = ["tab:green" if v <= LIMIT + 1e-6 else "tab:red" for v in summary["Rijeka-Velebit_real"]]
axes[0].bar(summary.index, summary["market_cost"] / 1e3, color="tab:blue")
axes[0].axhline(NODAL["cost"] / 1e3, color="black", ls="--", lw=1, label="nodal optimum")
axes[0].set(ylabel="market cost (thousand EUR/h)", title="What the market pays for its dispatch")
axes[1].bar(summary.index, summary["Rijeka-Velebit_real"], color=colours)
axes[1].axhline(LIMIT, color="black", ls="--", lw=1, label="line limit")
axes[1].set(ylabel="MW", title="Real flow on Rijeka-Velebit (red: overloaded)")
for ax in axes:
    ax.tick_params(axis="x", labelsize=8, rotation=15)
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

Only the nodal design is both secure and as cheap as security allows. The one-zone market looks cheapest but needs 10,956 €/h of redispatch to become secure. The two zonal fixes — an NTC and flow-based constraints on internal lines — cost more than the nodal optimum **and** still leave Rijeka–Velebit overloaded, so redispatch is needed on top. The common cause is that a zonal market only decides how much each zone produces, never **where** inside a zone; whenever congestion lies inside a zone, the zone's price cannot see it. Europe's answers are three: redispatch after the market (lesson 6), redrawing zone borders along the congestion (the exercise tries it), or nodal pricing — which several US markets use and which is discussed for Europe, against concerns about market liquidity and price differences within a country.

## Check your understanding

**Zones and redispatch**

1. The one-zone market is 10,956 EUR/h cheaper than the nodal dispatch. Is it really cheaper for society? Who pays for the difference?
2. Why can no NTC between North and South remove the overload on Rijeka–Velebit?

**Flow-based**

1. The flow-based market predicted exactly 400 MW on Rijeka–Velebit; the real flow was 447 MW. Which assumption failed?
2. Including internal lines in flow-based coupling is contested in Europe. Using this lesson's numbers, give one argument for and one against.